# Redis Index Klassifkationsmodell

Environment: redis

Indexierung der PDF-Dokumente im Redis Vector-Store für Retrieval mit dem trainiertem Klassifkationsmodell.

**Voraussetzung**

Environment: dski

Emebdding-Modell mittels vLLM über Rest-API zur Verfügung stellen und Redis-Server für Indexierung der Embedding-Vektoren starten (siehe setup.md).

**Indexierung**

Die Indexierung wird für die 245 Dokumente durchgeführt, auf welche sich die 397 gewählten Fragen aus dem Datensatz beziehen.

Die Blöcke der einzelnen Dokumente werden mit dem trainierten trainierten Modell klassifiziert. Anschließend werden alle Blöcke zwischen den erkannten Grenzen zusammengefügt. Für die erstellten Abschnitte werden durch das Embedding-Modell Embedding-Vektoren mit der Dimension 2048 erzeugt und diese im Vector-Store indexiert.

## Setup

### Pakete laden

In [1]:
import config

#import os
#import json
import math
import time
import hashlib

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq

from openai import OpenAI

import torch
from torch.utils.data import DataLoader

from datasets import DatasetDict, Dataset
from transformers import DataCollatorWithPadding, AutoTokenizer, AutoModelForSequenceClassification

import redis

from redis.commands.search.field import NumericField, TagField, TextField, VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.query import Query

### Variablen+Funktionen

In [2]:
# Redis Client
redis_client = redis.Redis(host=config.redis_client['host'], port=config.redis_client['port'], password=config.redis_client['password'], decode_responses=True)

# Masked-Language-Modell
model_id = 'FacebookAI/xlm-roberta-base'

# Embedding-Modell
embedding_model_id = 'nvidia/llama-nemotron-embed-1b-v2'
# Fenstergröße für Segmentierung
embedding_model_max_len = 6144#8192
# 1/3 der Fenstergröße
embedding_model_max_len_third = math.floor(embedding_model_max_len / 3)

# OpenAI Client
oai_client_embedding = OpenAI(
    # defaults to os.environ.get('OPENAI_API_KEY')
    api_key=config.openai_client['api_key'],
    base_url=config.openai_client['embedding']['base_url'],
)

### Daten laden

Beispiel-Fragen, Referenz-Antworten und PDF-URLs aus Datensatz laden.

In [3]:
# Answers
answers = pd.read_json(f'{config.dataset['path']}answers.json', orient='index', dtype_backend='numpy_nullable').rename(columns={0: 'answer'})
# Queries
queries = pd.read_json(f'{config.dataset['path']}queries.json', orient='index', dtype_backend='numpy_nullable')
# Relations
qrels = pd.read_json(f'{config.dataset['path']}qrels.json', orient='index', dtype_backend='numpy_nullable')

# PDF URLs
#pdf_urls = pd.read_json(f'{config.dataset['path']}pdf_urls.json', orient='index', dtype_backend='numpy_nullable').rename(columns={0: 'pdf_urls'})

pdf_urls_clean = pd.read_json(f'{config.dataset['path']}pdf_urls_clean.json', orient='index', dtype_backend='numpy_nullable')#.rename(columns={0: 'pdf_urls'})
pdf_urls = pdf_urls_clean.loc[pdf_urls_clean.values]

### Fragen selektieren

Fragen mit Quelle "Text" und "Text-Tabelle" und mindestens acht Wörtern in der Referenz-Antwort (Fragen mit kurzen Antworten wie "yes" ausschließen) für Ausführung selektieren

In [5]:
# Fragen und Antworten über Index joinen
queries_answers = queries.merge(answers, how='inner', left_index=True, right_index=True)

# Anzahl Wörter der Fragen 
queries_answers['answer_replace'] = queries_answers['answer'].str.replace(r'\p{P}+', '', regex=True).str.replace(r'[\n\t\s]+', ' ', regex=True)
queries_answers['answer_word_count'] = queries_answers['answer_replace'].str.split(' ').str.len()

# Fragen mit source=text|text-table und answer_word_count>=8
queries_text_text_table = queries_answers.loc[((queries_answers['source'] == 'text') | (queries_answers['source'] == 'text-table')) & (queries_answers['answer_word_count'] >= 8)]
    
queries_text_text_table = queries_text_text_table.sample(n=round(queries_text_text_table.shape[0] / 4), axis=0, random_state=42)

In [6]:
# Beziehung zwischen Frage und Dokument über qrels
# qrels/Dokumente für alle Frage wählen
qrels_select = qrels.loc[queries_text_text_table.index]
# qrels ausschließen, für die Dokumente (doc_id) nicht in pdf_urls (index)
qrels_select = qrels_select.drop(qrels_select.loc[~qrels_select['doc_id'].isin(pdf_urls.index)].index)

pdf_urls_select = pdf_urls.loc[qrels_select['doc_id'].unique()]

### Modell laden

Trainiertes MLM für Segmentierung der Dokumente laden

In [7]:
model_id_train ='./train/' + model_id
model_checkpoint = 'checkpoint-36978'

tokenizer = AutoTokenizer.from_pretrained(f'{model_id_train}/{model_checkpoint}')
model = AutoModelForSequenceClassification.from_pretrained(f'{model_id_train}/{model_checkpoint}', dtype=torch.float16).to('cuda')#torch_dtype

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Embedding-Modell Tokenizer

In [8]:
tokenizer_embedding = AutoTokenizer.from_pretrained(
    embedding_model_id, 
    trust_remote_code=True
)

### Dataset Preprocessing

In [9]:
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

### Segmentierung Klassifikationsmodell

Funktion zur Unterteilung des Text eines PDF-Dokument in Abschnitte mit Klassifikationsmodell und Überlappungen auf Tokenebene.

Abschnitte, welche die maximale Sequenzlänge von 8192 Token überschreiten, werden in Blöcken von 1/3 der maximalen Sequenzlänge mit 50% Überlappung zu beiden Seiten unterteilt.

In [10]:
def process_document(file_hash, file, pipeline, df_doc_processed, overlap=0.0):
    
    # Datensatz für Klassifizierung des Dokuments durch Modell erstellen
    dataset = DatasetDict({
        'test': Dataset.from_dict({'text': df_doc_processed['text'].tolist()})#, 'label': df_doc_processed['boundary']
    })

    # Datensatz Preprocessing
    tokenized_dataset = dataset.map(preprocess_function, batched=True)

    # Nur die für das Modell benötigten Spalten
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

    # DataCollator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='pt')

    # DataLoader
    test_loader = DataLoader(tokenized_dataset['test'], batch_size=1024, shuffle=False, collate_fn=data_collator)

    # Eval mode
    model.eval()
    
    predict_class = []
    
    time_start = time.time()

    # Vorhersage
    with torch.no_grad():
        for batch in test_loader:
            # Batch auf GPU verschieben
            batch = {
                key: value.to('cuda')
                for key, value in batch.items()
            }
    
            # Vorhersage
            outputs = model(**batch)
    
            # Klasse mit höchster Wahrscheinlichkeit wählen
            batch_predictions = torch.argmax(outputs.logits, dim=-1)
    
            # Direkt zurück auf CPU
            predict_class.extend(batch_predictions.cpu().tolist())

    # letzte Grenze
    boundary_last = 0
    predict_class_len = len(predict_class)

    # Liste für Inhalte der Blöcke
    blocks_content = []

    # Über Vorhersagen für Blöcke iterieren
    for i in range(predict_class_len):
        # Wenn Grenze vorhergesagt oder letzter Abschnitt
        if predict_class[i] == 1 or i + 1 == predict_class_len:
            # Wenn vorheriger Abschnitt keine Grenze
            if not (i > 0 and predict_class[i - 1] == 1):
                # Inhalte von letzter Grenze bis vor aktuelle Grenze zusammenfassen
                block_content = df_doc_processed.iloc[boundary_last:i]['text'].agg('\n\n'.join)
                # Anzahl der Token für Block ermitteln
                block_content_token = tokenizer_embedding(block_content)['input_ids'][1:]
                block_content_token_count = len(block_content_token)
                # Wenn Anzahl der Token maximale Länge von 6144 überschreitet
                if block_content_token_count > embedding_model_max_len:
                    # Blöcke in Teil-Blöcke mit Größe von 1/3 der Maximallänge mit 100% Überlappung unterteilen
                    # Anzahl Teilblöcke berechnen
                    block_content_parts = math.ceil(block_content_token_count / embedding_model_max_len_third)
                    for j in range(block_content_parts):
                        # Ende des letzten Blocks
                        block_content_end = (j + 1) * embedding_model_max_len_third
                        # Wenn Ende größer als Maximalänge von 6144
                        if block_content_end > block_content_token_count:
                            # Ende auf Maximalänge setzen
                            block_content_end = block_content_token_count
                        
                        # Inhalt des Blocks
                        block_content_part = ''

                        # Overlapping linke Seite des Blocks
                        if overlap > 0.0 and j > 0:
                            block_content_part+= tokenizer_embedding.decode(block_content_token[j * embedding_model_max_len_third - round(embedding_model_max_len_third * overlap): j * embedding_model_max_len_third]).strip() + ' '

                        # Block Inhalt hinzufügen
                        block_content_part+= tokenizer_embedding.decode(block_content_token[j * embedding_model_max_len_third: block_content_end]).strip()

                        # Overlapping rechte Seite des Blocks
                        if overlap > 0.0 and j + 1 < block_content_parts:
                            block_content_end_next = block_content_end + round(embedding_model_max_len_third * overlap)
                            if block_content_end_next > block_content_token_count:
                                block_content_end_next = block_content_token_count
                            block_content_part+= ' ' + tokenizer_embedding.decode(block_content_token[block_content_end: block_content_end_next]).strip()
                        
                        # Teil-Block zu Index hinzufügen
                        pipeline.json().set(
                            f'document:{file_hash}:{len(blocks_content)}', 
                            '$', 
                            {'document': file, 'content': block_content_part}#'page': 0, 
                        )
                        # Inhalt des Blocks zu Liste hinzufügen
                        blocks_content.append(block_content_part)
                else:
                    # Vollständigen Block zu Index hinzufügen
                    pipeline.json().set(
                        f'document:{file_hash}:{len(blocks_content)}', 
                        '$', 
                        {'document': file, 'content': block_content}
                    )
                    # Inhalt des Blocks zu Liste hinzufügen
                    blocks_content.append(block_content)
                # letzte Grenze auf aktuelle Grenze setzen
                boundary_last = i

    # Embeddings für Blöcke erzeugen
    embeddings = oai_client_embedding.embeddings.create(
        input=blocks_content,
        model=embedding_model_id
    )

    # Embeddings der Blöcke zu Einträgen hinzufügen
    # Über Blöcke iterieren
    for i in range(len(blocks_content)):
        pipeline.json().set(
            f'document:{file_hash}:{i}', 
            '$.content_embeddings', embeddings.data[i].embedding[:embedding_dimension]
        )

    res = pipeline.execute()

## Indexierung

### PDF-Dokumente indexieren

Alle PDF-Dokumente, auf welche sich Fragen beziehen, einlesen und abschnittsweise in redis indexieren

In [11]:
# Index löschen
redis_client.flushdb()
#redis_client.keys('document:*')

# Dimension der Embedding-Vektoren
embedding_dimension = 2048

time_start = time.time()

# Redis Pipeline erstellen
pipeline = redis_client.pipeline()

# Über pdf_urls/Dokumente iterieren
for idx_doc in range(pdf_urls_select.shape[0]):#[10]:#
    doc_id = pdf_urls_select.iloc[idx_doc].name
    print(f'{idx_doc + 1} {doc_id}')

    # Hashwert der Dokument-ID für Keys berechnen
    h = hashlib.sha256()
    h.update(doc_id.encode())

    # Dataframe mit vorverarbeiteten Daten laden
    df_doc_processed = pq.read_table(f'{config.dataset['path']}pdfs_df_process/{doc_id}.parquet').to_pandas()
    # Dokument verarbeiten
    process_document(h.hexdigest(), doc_id, pipeline, df_doc_processed, overlap=1.0)

print(time.time() - time_start)

1 2406.06650v2


Map:   0%|          | 0/103 [00:00<?, ? examples/s]

2 2412.05495v6


Map:   0%|          | 0/510 [00:00<?, ? examples/s]

3 2412.15929v3


...

244 2402.03637v2


Map:   0%|          | 0/957 [00:00<?, ? examples/s]

245 2410.05454v2


Map:   0%|          | 0/268 [00:00<?, ? examples/s]

596.3589010238647


In [12]:
len(redis_client.keys('document:*'))

5556

### Redis Vector-Index

Redis Vector-Index erstellen

In [13]:
#embedding_dimension = 2048

schema = (
    TextField('$.document', no_stem=True, sortable=False, as_name='document'),#, no_index=True
    #NumericField('$.page', as_name='page'),
    TextField('$.content', no_stem=True, sortable=False, as_name='content'),
    VectorField(
        '$.content_embeddings',
        'FLAT',#'SVS-VAMANA',
        {
            'TYPE': 'FLOAT16',
            'DIM': embedding_dimension,
            'DISTANCE_METRIC': 'IP', #'COSINE',
        },
        as_name='vector',
    ),
)

definition = IndexDefinition(prefix=['document:'], index_type=IndexType.JSON)
res = redis_client.ft('idx:document').create_index(fields=schema, definition=definition)

In [18]:
info = redis_client.ft('idx:document').info()
print(f'Abschnitte im Index: {info['num_docs']}')
print(f'hash_indexing_failures: {info['hash_indexing_failures']}')

Abschnitte im Index: 5556
hash_indexing_failures: 0


### Index Abfrage

Abfrage der drei ähnlichsten Abschnitte für eine Beispiel-Frage

In [19]:
index = 0

# Modell-Eingabe
queries = [
    queries_text_text_table.iloc[index]['query']
]

# Embedding zu Eingabe erzeugen
embeddings = oai_client_embedding.embeddings.create(
    input=queries,
    model=embedding_model_id,
)

# Vector-Store Retrieval
query = (
    Query('(*)=>[KNN 3 @vector $query_vector AS vector_score]')
     .sort_by('vector_score', asc=False)
     .return_fields('vector_score', 'id', 'document', 'content')#, 'vector'#, 'page'
     .dialect(2)
)

result = redis_client.ft('idx:document').search(
    query,
    {
        'query_vector': np.array([embeddings.data[0].embedding[:embedding_dimension]], dtype=np.float16).tobytes()
    }
)

result.docs#[0]['vector']

[Document {'id': 'document:c2f85fa5d4dfedb039ca9a3881169c139b422d6fae384a2e8d07c5aba5cbd7a9:21', 'payload': None, 'vector_score': '0.727615952492', 'document': '2404.11929v3', 'content': 'Acknowledgement\n\nFunding: This work was supported in part by Hankuk University of Foreign Studies (HUFS) Basic Research Fund, South Korea (No. 20231140001) and the National Research Foundation of Korea (NRF) grant funded by the Korea government (MSIT) (No. 2022R1F1A1064530) and grant No. 02–2023-0011 from the Seoul National University Bundang Hospital Research Fund (corresponding authors: Il Dong Yun and Yun Jung Bae).'},
 Document {'id': 'document:cd9893568c93927bb833ae9f0b78c6918e4680d7f67b065cd298685de319584f:0', 'payload': None, 'vector_score': '0.701801896095', 'document': '2406.06650v2', 'content': 'Assessing the risk of recurrence in early-stage breast cancer through H&E stained whole slide images\n\nGeongyu Lee1*, Joonho Lee1, Tae-Yeong Kwak1, Sun Woo Kim1, Youngmee Kwon2, Chungyeul Kim3*, H

In [27]:
print('query')
print(queries_text_text_table.iloc[index]['query'])
print('reference answer')
print(queries_text_text_table.iloc[index]['answer'])

query
Is Tae-Yeong Kwak associated with Deep Bio Inc.?
reference answer
Yes, Tae-Yeong Kwak is associated with Deep Bio Inc., serving as CTO.


In [28]:
# Dokument
#df_doc = pq.read_table(f'{config.dataset['path']}pdfs_df/{qrels.loc[queries_text_text_table.iloc[index].name, 'doc_id']}.parquet').to_pandas()
#print(df_doc['text'].agg('\n\n'.join))